# Step 2 - Flight-Only Feature Refinement

This notebook keeps the model flight-only, adds stronger features, uses a time split, and compares models.


In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

## Load And Clean Data


In [15]:
data_dir = Path("../data/raw/flight/2022")
csv_files = sorted(data_dir.glob("2022-0[1-3]_flight.csv"))

cols = [
    "Month",
    "FlightDate",
    "DayOfWeek",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepDelay",
    "Cancelled"
]

dfs = []

for file in csv_files:
    df_part = pd.read_csv(file, usecols=cols)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)
df.shape

(1598468, 9)

In [16]:
df = df[df["Cancelled"] == 0].copy()

df = df.dropna(subset=[
    "DepDelay",
    "FlightDate",
    "CRSDepTime",
    "Reporting_Airline",
    "Origin",
    "Dest"
])

df = df.sample(n=200_000, random_state=42)

df["Delay"] = (df["DepDelay"] > 15).astype(int)

df = df.drop(columns=["DepDelay"])

df["CRSDepTime"] = df["CRSDepTime"].astype(int)
df["dep_hour"] = df["CRSDepTime"] // 100
df["FlightDate"] = pd.to_datetime(df["FlightDate"])

dep_minutes = (df["CRSDepTime"] // 100) * 60 + (df["CRSDepTime"] % 100)
df["scheduled_departure"] = df["FlightDate"] + pd.to_timedelta(dep_minutes, unit="m")

df.shape

(200000, 11)

## Add Flight Features


In [17]:
enhanced_df = df.copy()

enhanced_df["route"] = enhanced_df["Origin"] + "_" + enhanced_df["Dest"]

enhanced_df["weather_station"] = "K" + enhanced_df["Origin"]

enhanced_df["is_weekend"] = enhanced_df["DayOfWeek"].isin([6, 7]).astype(int)

enhanced_df["time_of_day_bin"] = pd.cut(
    enhanced_df["dep_hour"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=["overnight", "morning", "afternoon", "evening", "night"]
)

enhanced_df[["FlightDate", "scheduled_departure", "Origin", "weather_station", "Dest", "route", "DayOfWeek", "is_weekend", "dep_hour", "time_of_day_bin"]].head()

,FlightDate,scheduled_departure,Origin,weather_station,Dest,route,DayOfWeek,is_weekend,dep_hour,time_of_day_bin
1520653,2022-03-06,2022-03-06 13:16:00,DFW,KDFW,ABQ,DFW_ABQ,7,1,13,afternoon
344435,2022-01-18,2022-01-18 14:15:00,SAN,KSAN,PHX,SAN_PHX,2,0,14,afternoon
399849,2022-01-28,2022-01-28 13:29:00,JFK,KJFK,BWI,JFK_BWI,5,0,13,afternoon
210583,2022-01-27,2022-01-27 11:10:00,LAX,KLAX,KOA,LAX_KOA,4,0,11,morning
1241754,2022-03-10,2022-03-10 20:12:00,DEN,KDEN,ORD,DEN_ORD,4,0,20,evening


## Time-Based Split

Train on January and February. Test on March.


In [18]:
train_df = enhanced_df[enhanced_df["Month"].isin([1, 2])].copy()
test_df = enhanced_df[enhanced_df["Month"] == 3].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train delay rate:", train_df["Delay"].mean())
print("Test delay rate:", test_df["Delay"].mean())

Train shape: (127423, 15)
Test shape: (72577, 15)
Train delay rate: 0.1900284877926277
Test delay rate: 0.20864736762335176


## Top 20 Origin Airports

This list is used for the first weather version.


In [19]:
top_20_origin_airports = (
    enhanced_df.groupby(["Origin", "weather_station"], observed=True)
    .agg(
        flights=("Delay", "size"),
        delay_rate=("Delay", "mean")
    )
    .reset_index()
    .sort_values("flights", ascending=False)
    .head(20)
)

top_20_origin_airports

,Origin,weather_station,flights,delay_rate
21,ATL,KATL,9605,0.165018
94,DFW,KDFW,7999,0.173272
93,DEN,KDEN,7806,0.313605
246,ORD,KORD,7757,0.197499
71,CLT,KCLT,6003,0.152757
188,LAX,KLAX,5807,0.157224
260,PHX,KPHX,5254,0.177198
197,LGA,KLGA,5107,0.216174
186,LAS,KLAS,4941,0.212710
305,SEA,KSEA,4881,0.169023


In [20]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

top_origin_path = processed_dir / "top_20_origin_weather_stations.csv"
top_20_origin_airports.to_csv(top_origin_path, index=False)

top_origin_path

WindowsPath('../data/processed/top_20_origin_weather_stations.csv')

## Historical Delay Rate Features


In [21]:
def add_delay_rate_feature(train_data, test_data, group_col, target_col, new_col):
    """Add historical delay rates using training data only to avoid leakage."""
    global_rate = train_data[target_col].mean()
    rates = train_data.groupby(group_col)[target_col].mean()

    train_data[new_col] = train_data[group_col].map(rates).fillna(global_rate)
    test_data[new_col] = test_data[group_col].map(rates).fillna(global_rate)

    return train_data, test_data


train_df, test_df = add_delay_rate_feature(
    train_df,
    test_df,
    group_col="Origin",
    target_col="Delay",
    new_col="origin_delay_rate"
)

train_df, test_df = add_delay_rate_feature(
    train_df,
    test_df,
    group_col="Reporting_Airline",
    target_col="Delay",
    new_col="airline_delay_rate"
)

train_df, test_df = add_delay_rate_feature(
    train_df,
    test_df,
    group_col="route",
    target_col="Delay",
    new_col="route_delay_rate"
)

train_df[["Origin", "Reporting_Airline", "route", "origin_delay_rate", "airline_delay_rate", "route_delay_rate"]].head()

,Origin,Reporting_Airline,route,origin_delay_rate,airline_delay_rate,route_delay_rate
344435,SAN,WN,SAN_PHX,0.129154,0.218218,0.107527
399849,JFK,YX,JFK_BWI,0.231337,0.171872,0.066667
210583,LAX,HA,LAX_KOA,0.151314,0.146203,0.083333
324523,SMF,WN,SMF_LAS,0.093784,0.218218,0.131148
973753,IND,WN,IND_AUS,0.190231,0.218218,0.161290


## Model Comparison

This compares Logistic Regression, Random Forest, and Hist Gradient Boosting on the refined flight-only feature set.


In [22]:
enhanced_numeric_features = [
    "DayOfWeek",
    "dep_hour",
    "is_weekend",
    "origin_delay_rate",
    "airline_delay_rate",
    "route_delay_rate"
]

enhanced_categorical_features = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "route",
    "time_of_day_bin"
]

enhanced_features = enhanced_numeric_features + enhanced_categorical_features

X_train = train_df[enhanced_features]
y_train = train_df["Delay"]
X_test = test_df[enhanced_features]
y_test = test_df["Delay"]

ohe_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="most_frequent"), enhanced_numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), enhanced_categorical_features)
])

ordinal_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="most_frequent"), enhanced_numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ]), enhanced_categorical_features)
])

In [23]:
candidate_models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", ohe_preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", ohe_preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=50,
            max_depth=12,
            min_samples_leaf=5,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]),
    "Hist Gradient Boosting": Pipeline([
        ("preprocessor", ordinal_preprocessor),
        ("classifier", HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.08,
            max_leaf_nodes=31,
            random_state=42,
            class_weight="balanced"
        ))
    ])
}

In [24]:
model_results = []

for model_name, candidate_model in candidate_models.items():
    candidate_model.fit(X_train, y_train)
    y_pred = candidate_model.predict(X_test)

    model_results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_delay": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_delay": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_delay": f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    })

    print("=" * 80)
    print(model_name)
    print(classification_report(y_test, y_pred, zero_division=0))
    print(confusion_matrix(y_test, y_pred))

pd.DataFrame(model_results).sort_values("f1_delay", ascending=False)

Logistic Regression
              precision    recall  f1-score   support

           0       0.85      0.63      0.72     57434
           1       0.29      0.57      0.38     15143

    accuracy                           0.62     72577
   macro avg       0.57      0.60      0.55     72577
weighted avg       0.73      0.62      0.65     72577

[[36124 21310]
 [ 6575  8568]]
Random Forest
              precision    recall  f1-score   support

           0       0.85      0.64      0.73     57434
           1       0.30      0.58      0.40     15143

    accuracy                           0.63     72577
   macro avg       0.58      0.61      0.56     72577
weighted avg       0.74      0.63      0.66     72577

[[36732 20702]
 [ 6285  8858]]
Hist Gradient Boosting
              precision    recall  f1-score   support

           0       0.85      0.65      0.74     57434
           1       0.30      0.56      0.39     15143

    accuracy                           0.63     72577
   macro 

,model,accuracy,precision_delay,recall_delay,f1_delay
1,Random Forest,0.628160,0.299662,0.584957,0.396304
2,Hist Gradient Boosting,0.632873,0.298846,0.564221,0.390735
0,Logistic Regression,0.615787,0.286766,0.565806,0.380622
